# Notebook 3: Visualization & Analysis

This notebook explores the data produced by Notebooks 1 and 2 through visualization. The central question is:

> **Do organelle properties vary systematically along the portal-to-central vein axis?**

This spatial pattern — called **liver zonation** — is well established at the gene-expression level, but `liv_zones` lets you measure it directly at the organelle level in fluorescence images.

### What this notebook covers

1. An annotated overview of all cells in the acinus
2. Trend plots showing how organelle properties change along the acinus axis
3. A per-cell view combining raw image channels and segmentation overlays
4. A correlation heatmap between cell-level features

---

## Biological motivation

Hepatocytes (liver cells) near the **portal vein** receive nutrient-rich, oxygenated blood first. Those near the **central vein** receive blood that has already been processed — lower in oxygen and nutrients. This gradient causes cells to specialize:

- **Periportal cells** (near portal vein, position ≈ +1): higher oxidative metabolism, more elongated mitochondria
- **Pericentral cells** (near central vein, position ≈ −1): higher lipid metabolism, more lipid droplets

`liv_zones` quantifies these gradients from microscopy images of a single acinus.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

from liv_zones.ascini import plot_properties, plot_cell, plot_ascinus_annotated

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH  = "sample_data/output"
IMAGE_PATH = "sample_data/sample_image.tif"

# Load the cell-level data produced in Notebook 2
cell_props = pd.read_csv(f"{DATA_PATH}/average_properties_per_cell.csv", index_col=0)
print(f"Loaded {len(cell_props)} cells with {len(cell_props.columns)} features")

## 2. Acinus overview

`plot_ascinus_annotated()` renders the cell segmentation mask with each cell labeled by its integer ID. This is useful for identifying which cell IDs correspond to which regions of the acinus before calling `plot_cell()` on specific cells of interest.

The function writes a `labeled_cells.png` to the data directory and also displays the image.

In [ ]:
plot_ascinus_annotated(DATA_PATH)

# Display the saved image inline
img = plt.imread(f"{DATA_PATH}/labled_cells.png")
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img)
ax.axis("off")
ax.set_title("Acinus — cells labeled by ID", fontsize=13)
plt.tight_layout()
plt.show()

print("\nCell IDs range from", cell_props['label'].min(),
      "to", cell_props['label'].max())

## 3. Spatial trends along the acinus

`plot_properties()` generates a scatter plot for every column in `average_properties_per_cell.csv` (except a few index columns), sorted by acinus position. A rolling mean is overlaid to make trends easier to see.

The function saves all plots to `DATA_PATH/ascini_trends/`. Here we show a curated selection inline.

In [ ]:
# Generate and save all trend plots
plot_properties(
    path=DATA_PATH,
    colors=["steelblue", "coral"],
)

print("All trend plots saved to", f"{DATA_PATH}/ascini_trends/")

### Reproducing key plots inline

Below we re-create trend plots for a curated set of features to display them here. Cells are sorted by `ascini_position`, and the coral line shows a rolling mean (window = 20 cells, minimum 10).

In [ ]:
def trend_plot(ax, df, prop, title, ylabel, color="steelblue"):
    """Plot a single property vs. acinus position with a rolling trend line."""
    sorted_df = df.sort_values("ascini_position")
    position = sorted_df["ascini_position"]
    values   = sorted_df[prop]
    trend    = values.rolling(20, min_periods=10, center=True).mean()

    ax.scatter(position, values, c=color, s=20, alpha=0.6, label="Individual cells")
    ax.plot(position, trend, c="coral", linewidth=2.5, label="Rolling mean")
    ax.set_xlabel("Acinus position\n(−1 = central vein, +1 = portal vein)", fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.axvline(0, color="black", linestyle=":", alpha=0.4)
    ax.legend(fontsize=8)


# Choose features to display — skip any that don't exist in this dataset
features_to_show = [
    ("mito_density",         "Mito density\n(mitochondria / µm²)",     "steelblue"),
    ("mito_aspect_ratio",    "Mito aspect ratio\n(elongation)",         "#2ecc71"),
    ("ld_density",           "Lipid droplet density\n(droplets / µm²)", "#3498db"),
    ("peroxisome_density",   "Peroxisome density\n(perox / µm²)",       "#f39c12"),
]
features_to_show = [(f, label, c)
                    for f, label, c in features_to_show
                    if f in cell_props.columns]

n = len(features_to_show)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=False)
if n == 1:
    axes = [axes]

for ax, (feat, ylabel, color) in zip(axes, features_to_show):
    trend_plot(ax, cell_props, feat, feat, ylabel, color)

plt.suptitle("Organelle properties along the portal-to-central axis",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### What to look for

- **Mito density / aspect ratio:** These often increase toward the portal end (+1) in healthy liver, reflecting higher oxidative metabolism.
- **Lipid droplet density:** Often peaks toward the central end (−1) where lipid synthesis is more active.
- **Peroxisome density:** Typically higher periportally, consistent with fatty acid oxidation.

The rolling mean (coral line) smooths out cell-to-cell variability to reveal the underlying gradient.

## 4. Visualising individual cells

`plot_cell()` shows a 2 × 3 grid for a chosen cell:

| Row | Col 0 | Col 1 | Col 2 |
|:---|:---|:---|:---|
| **Top** | Actin (raw) | Mito (raw) | Lipid (raw) |
| **Bottom** | *(empty)* | Mito mask | Lipid mask |

Use the annotated acinus image from Section 2 to choose a cell ID of interest. Here we show cells near the portal and central ends of the acinus to highlight differences.

In [ ]:
# Sort cells by acinus position
sorted_cells = cell_props.sort_values("ascini_position")

# Pick a cell near the central vein (most negative position) and portal vein (most positive)
central_cell_id = int(sorted_cells.iloc[0]["label"])
portal_cell_id  = int(sorted_cells.iloc[-1]["label"])

print(f"Most central cell: ID {central_cell_id},  "
      f"position = {sorted_cells.iloc[0]['ascini_position']:.3f}")
print(f"Most portal  cell: ID {portal_cell_id},  "
      f"position = {sorted_cells.iloc[-1]['ascini_position']:.3f}")

In [ ]:
# Visualise the most central cell
print(f"Plotting cell {central_cell_id} (near central vein)...")
plot_cell(
    image_path=IMAGE_PATH,
    mask_path=DATA_PATH,
    cell_num=central_cell_id,
)

# Display inline
img = plt.imread(f"{DATA_PATH}/cell_{central_cell_id}_segmentation.png")
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img)
ax.axis("off")
ax.set_title(f"Cell {central_cell_id} — near central vein", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the most portal cell
print(f"Plotting cell {portal_cell_id} (near portal vein)...")
plot_cell(
    image_path=IMAGE_PATH,
    mask_path=DATA_PATH,
    cell_num=portal_cell_id,
)

img = plt.imread(f"{DATA_PATH}/cell_{portal_cell_id}_segmentation.png")
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img)
ax.axis("off")
ax.set_title(f"Cell {portal_cell_id} — near portal vein", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Correlation analysis

The correlation heatmap shows **R² values** (coefficient of determination) between pairs of cell-level features. Only correlations with p < 0.01 are shown; non-significant values are displayed as NaN (white).

This is a cleaned-up, parameterized version of the logic in `correlationMatrix.py`.

### How to read it

- **Red** = strong positive correlation (features increase together)
- **Blue** = strong negative correlation (one increases as the other decreases)
- **White** = no significant linear relationship (p ≥ 0.01)

The diagonal is always 1.0 (a feature is perfectly correlated with itself).

In [ ]:
# Select a curated subset of columns for the correlation matrix.
# Add or remove columns to focus on what interests you.
candidate_cols = [
    "ascini_position",
    "area",
    # Mitochondria
    "mito_density",
    "mito_avg_area",
    "mito_aspect_ratio",
    "mito_percent_total_area",
    "percent_type_1_mito",
    "percent_type_3_mito",
    # Lipid droplets
    "ld_density",
    "ld_avg_area",
    "ld_percent_total_area",
    "percent_type_1_ld",
    "percent_type_3_ld",
    # Peroxisomes
    "peroxisome_density",
    "peroxisome_avg_area",
    "peroxisome_percent_total_area",
]
# Keep only columns that actually exist
properties = [c for c in candidate_cols if c in cell_props.columns]
print(f"Using {len(properties)} features for correlation matrix")

In [ ]:
# Drop rows with any NaN in the selected columns
clean = cell_props[properties].dropna()
print(f"Cells with complete data: {len(clean)} / {len(cell_props)}")

n = len(properties)
rvals = np.full((n, n), np.nan)

for i in range(n):
    for j in range(i + 1):  # lower triangle only
        result = scipy.stats.linregress(clean[properties[i]], clean[properties[j]])
        r, p   = result.rvalue, result.pvalue
        if p < 0.01:
            rvals[i, j] = r   # store R (not R²) so direction is preserved
        # else leave as NaN

# Symmetrise for display
rvals_sym = np.where(np.isnan(rvals), rvals, rvals)  # already lower-tri; mirror it
for i in range(n):
    for j in range(i + 1, n):
        rvals_sym[i, j] = rvals_sym[j, i]

# ── Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(8, n * 0.5 + 2), max(7, n * 0.5 + 1)))

cmap = plt.get_cmap("coolwarm").copy()
cmap.set_bad("white")          # NaN → white

im = ax.imshow(rvals_sym, vmin=-1, vmax=1, cmap=cmap, aspect="auto")
plt.colorbar(im, ax=ax, label="Pearson R  (p < 0.01 only)", fraction=0.046)

tick_positions = np.arange(n)
ax.set_xticks(tick_positions)
ax.set_xticklabels(properties, rotation=45, ha="right", fontsize=8)
ax.set_yticks(tick_positions)
ax.set_yticklabels(properties, fontsize=8)

ax.set_title("Feature correlation matrix  (R, p < 0.01;  white = not significant)",
             fontsize=11, pad=12)
plt.tight_layout()
plt.savefig(f"{DATA_PATH}/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {DATA_PATH}/correlation_matrix.png")

### Interpreting the matrix

Some correlations to look for:

- **`ascini_position` vs. mito features** — a strong positive R indicates that the feature increases toward the portal end; negative means it increases toward the central end.
- **`mito_density` vs. `ld_density`** — often anti-correlated, reflecting the metabolic trade-off between oxidative and lipid-synthesis zones.
- **`percent_type_3_mito` vs. `ascini_position`** — if positive, elongated mitochondria are enriched periportally.

## 6. Next steps

Congratulations — you have completed the full `liv_zones` tutorial!

Here are some directions to explore next:

- **Full feature list:** See `docs/source/features.rst` for a complete description of every column in the output CSVs.

- **Multi-stack experiments:** Use `liv_zones.track_z_pos` to link cell IDs across z-stacks in a 3D acquisition (see `liv_zones/track_z_pos.py`).

- **Batch processing:** `run.py` in the package root shows how to loop over multiple images and save results in parallel.

- **API reference:** Full function signatures and docstrings are at `docs/source/api.rst` and in the online Sphinx documentation.

- **Adjusting classification thresholds:** The mitochondrial type split values (`mito_aspect_split`, `ld_area_split`) can be passed directly to `organelle_features()` if the defaults don't fit your data.